# LECTURE 14
**Topic:** Circular Shift and Reversal; Circular Conjugate Symmetry of V and W; DFT Properties 3 & 4  
**Textbook References:** Sections 3.2.2, 3.3.2, 3.4

---

## Key Points

- $Px$: circular shift **down** by 1 position. $P^m$: shift by $m$. $P^N=I$.
- $Rx$: circular reversal $[x[0],x[N-1],\ldots,x[1]]^T$. $R^2=I$.
- $F^k$: diagonal matrix, multiplies entry $n$ of $x$ by $e^{jk(2\pi/N)n}$ (the $k$-th Fourier sinusoid).
- Columns of $V$: $v^{(N-k)}=(v^{(k)})^*$, i.e., $W=V^*=VR=RV$.
- **DFT 3:** $x^* \leftrightarrow RX^*$ &emsp; **DFT 4:** $Rx \leftrightarrow RX$


---
## Theory and Examples

### 1. Circular Shift Matrix $P$

$$Px: [x[0],x[1],\ldots,x[N-1]]^T \mapsto [x[N-1],x[0],\ldots,x[N-2]]^T$$

$$P=\begin{bmatrix}0&0&\cdots&0&1\\1&0&\cdots&0&0\\0&1&\cdots&0&0\\\vdots&&\ddots&&\vdots\\0&0&\cdots&1&0\end{bmatrix}$$

$P^m$ = shift by $m$ positions; $P^N=I$.

### 2. Circular Reversal Matrix $R$

$$Rx: [x[0],x[1],\ldots,x[N-1]]^T \mapsto [x[0],x[N-1],\ldots,x[1]]^T$$

$R^2=I$ (reversing twice gives back the original).

### 3–4. Modulation Matrix $F^k$

$F=\text{diag}(1,v,v^2,\ldots,v^{N-1})$ where $v=e^{j2\pi/N}$.

$(F^k x)[n] = e^{jk(2\pi/N)n}\,x[n]$ — multiplies each entry by the corresponding entry of $v^{(k)}$.

**Question:** For even $N$, $y[n]=(-1)^n x[n]$. Since $(-1)^n=e^{j\pi n}=v^{N/2 \cdot n}$, we have $y=F^{N/2}x$.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

N = 6

# Build P, R, F matrices
def make_P(N):
    P = np.zeros((N,N))
    P[0,N-1]=1
    for i in range(1,N): P[i,i-1]=1
    return P

def make_R(N):
    R = np.zeros((N,N))
    R[0,0]=1
    for i in range(1,N): R[i,N-i]=1
    return R

def make_F(N, k=1):
    v = np.exp(1j*2*np.pi/N)
    return np.diag([v**(k*n) for n in range(N)])

P = make_P(N); R = make_R(N); F = make_F(N,1)
x = np.arange(N, dtype=float)

print(f'x = {x}')
print(f'Px = {P@x}  (circular shift down by 1)')
print(f'P^2 x = {(P@P)@x}  (circular shift down by 2)')
print(f'Rx = {R@x}  (circular reversal)')
print(f'R^2 x = {(R@R)@x}  (should equal x)')

# Verify P^N = I
PN = np.linalg.matrix_power(P,N)
print(f'\nP^N = I? {np.allclose(PN, np.eye(N))}')
print(f'R^2 = I? {np.allclose(R@R, np.eye(N))}')

x = [0. 1. 2. 3. 4. 5.]
Px = [5. 0. 1. 2. 3. 4.]  (circular shift down by 1)
P^2 x = [4. 5. 0. 1. 2. 3.]  (circular shift down by 2)
Rx = [0. 5. 4. 3. 2. 1.]  (circular reversal)
R^2 x = [0. 1. 2. 3. 4. 5.]  (should equal x)

P^N = I? True
R^2 = I? True


In [2]:
# Express x^(1) through x^(6) in terms of P, R, F
x = np.array([0,1,2,3,4,5], dtype=float)
v = np.exp(1j*2*np.pi/N)
F6 = make_F(N,1); P6 = make_P(N); R6 = make_R(N)

print('=== Express x^(1) through x^(6) ===')
print(f'x = {x}')

# x^(1) = [4,5,0,1,2,3] = P^4 x (circular shift by 4)
x1 = np.array([4,5,0,1,2,3], dtype=float)
P4 = np.linalg.matrix_power(P6,4)
print(f'x^(1)=[4,5,0,1,2,3] = P^4 x: {P4@x}  match={np.allclose(P4@x,x1)}')

# x^(2) = [4,3,2,1,0,5]: This is P^4 * R on x... let's check
x2 = np.array([4,3,2,1,0,5], dtype=float)
# x^(2) = [4,3,2,1,0,5]. Reverse x=[0,1,2,3,4,5] -> [0,5,4,3,2,1], then shift? 
# Actually x^(2) = P*R*x gives shift by 1 of reversal
# Reversal: [0,5,4,3,2,1], P^1 shift: [1,0,5,4,3,2] - no
# Let's find by checking: x^(2)[0]=4=x[4], x^(2)[1]=3=x[3]... reversed starting at n=4
# = P^4 R x
candidate = P4 @ R6 @ x
print(f'x^(2)=[4,3,2,1,0,5] = P^4*R*x: {candidate}  match={np.allclose(candidate,x2)}')

# x^(3) = [0,6,6,6,6,6] = x + F1*x (entry-wise: x[n] + v^n*x[n]... no)
# x^(3) = x + P^(-1)x = x + P^5 x = [0+5, 1+0, 2+1, 3+2, 4+3, 5+4]=[5,1,3,5,7,9] - no
# 0+0=0; 1+5=6; 2+4=6; 3+3=6; 4+2=6; 5+1=6. So x^(3) = x + Rx (but Rx=[0,5,4,3,2,1])
x3 = np.array([0,6,6,6,6,6], dtype=float)
candidate3 = x + R6@x
print(f'x^(3)=[0,6,6,6,6,6] = x + R*x: {candidate3}  match={np.allclose(candidate3,x3)}')

# x^(4) = [0,-4,-2,0,2,4]: x^(4)[n] = x[n]*(-1+1)... hmm. 
# x^(4) = x - Rx: [0-0,1-5,2-4,3-3,4-2,5-1]=[0,-4,-2,0,2,4]
x4 = np.array([0,-4,-2,0,2,4], dtype=float)
candidate4 = x - R6@x
print(f'x^(4)=[0,-4,-2,0,2,4] = x - R*x: {candidate4}  match={np.allclose(candidate4,x4)}')

# x^(5) = [0,-1,2,-3,4,-5] = F^(N/2) * x = diag(1,-1,1,-1,1,-1)*x (since v^(3n)=(-1)^n)
x5 = np.array([0,-1,2,-3,4,-5], dtype=float)
FN2 = make_F(N, N//2)  # F^(N/2): (-1)^n diagonal
candidate5 = (FN2 @ x).real
print(f'x^(5)=[0,-1,2,-3,4,-5] = F^(N/2)*x: {candidate5}  match={np.allclose(candidate5,x5)}')

# x^(6) = [0,2,0,6,0,10] = x + F^(N/2)*x (even-indexed doubled)
x6 = np.array([0,2,0,6,0,10], dtype=float)
candidate6 = x + (FN2@x).real
print(f'x^(6)=[0,2,0,6,0,10] = x + F^(N/2)*x: {candidate6}  match={np.allclose(candidate6,x6)}')

=== Express x^(1) through x^(6) ===
x = [0. 1. 2. 3. 4. 5.]
x^(1)=[4,5,0,1,2,3] = P^4 x: [2. 3. 4. 5. 0. 1.]  match=False
x^(2)=[4,3,2,1,0,5] = P^4*R*x: [4. 3. 2. 1. 0. 5.]  match=True
x^(3)=[0,6,6,6,6,6] = x + R*x: [0. 6. 6. 6. 6. 6.]  match=True
x^(4)=[0,-4,-2,0,2,4] = x - R*x: [ 0. -4. -2.  0.  2.  4.]  match=True
x^(5)=[0,-1,2,-3,4,-5] = F^(N/2)*x: [ 0. -1.  2. -3.  4. -5.]  match=True
x^(6)=[0,2,0,6,0,10] = x + F^(N/2)*x: [ 0.00000000e+00  0.00000000e+00  4.00000000e+00 -8.88178420e-16
  8.00000000e+00 -1.77635684e-15]  match=False


---
### 6–9. Circular Conjugate Symmetry of V and W

The $k$-th and $(N-k)$-th columns of $V$ are complex conjugates:
$$v^{(N-k)} = (v^{(k)})^*, \quad k=1:N-1$$

Using circular reversal: $V^* = W = VR = RV$ and $V = W^* = WR = RW$.

**Consequence:** $RWR=W$ and $RVR=V$.

### 11. DFT 3: Conjugation in time → Conjugation + Reversal in frequency

If $y=x^*$, then $Y=Wy=Wx^*=RW^*x^*=R(Wx)^*=RX^*$

**Summary:** $x^* \leftrightarrow RX^*$

### 12. DFT 4: Circular Reversal in time → Circular Reversal in frequency

If $y=Rx$, then $Y=WRx=RWx=RX$

**Summary:** $Rx \leftrightarrow RX$


In [3]:
# Verify DFT 3 and DFT 4 numerically
np.random.seed(7)
N = 6
x = np.random.randn(N) + 1j*np.random.randn(N)
X = np.fft.fft(x)
R6 = make_R(N)

# DFT 3: x* <-> R X*
y3 = x.conj()
Y3_fft = np.fft.fft(y3)
Y3_formula = R6 @ X.conj()
print('DFT 3: x* <-> R*X*')
print(f'  fft(x*) = {Y3_fft.round(4)}')
print(f'  R*X*    = {Y3_formula.round(4)}')
print(f'  Match? {np.allclose(Y3_fft, Y3_formula)}')

# DFT 4: Rx <-> RX
y4 = R6 @ x
Y4_fft = np.fft.fft(y4)
Y4_formula = R6 @ X
print('\nDFT 4: R*x <-> R*X')
print(f'  fft(Rx) = {Y4_fft.round(4)}')
print(f'  R*X     = {Y4_formula.round(4)}')
print(f'  Match? {np.allclose(Y4_fft, Y4_formula)}')

# Example 13: Use known DFT pair to get two new pairs
# [1, j, -2, 2j] <-> [-1+3j, 2, -1-3j, 4]
x_known = np.array([1, 1j, -2, 2j])
X_known = np.fft.fft(x_known)
print('\nExample 13:')
print(f'x  = {x_known}')
print(f'X  = {X_known.round(4)}')
print(f'Expected: [-1+3j, 2, -1-3j, 4]')

R4 = make_R(4)
# DFT 3: x* <-> R*X*
print(f'\nx* = {x_known.conj()}')
print(f'R*X* = {(R4@X_known.conj()).round(4)}')
print(f'fft(x*) = {np.fft.fft(x_known.conj()).round(4)}')
# DFT 4: Rx <-> RX
Rx = R4@x_known
print(f'\nRx = {Rx}')
print(f'R*X = {(R4@X_known).round(4)}')
print(f'fft(Rx) = {np.fft.fft(Rx).round(4)}')

DFT 3: x* <-> R*X*
  fft(x*) = [ 0.8781+0.9344j  1.3772+1.4543j  5.5021-0.2497j  0.9908-1.7171j
 -0.086 -2.4836j  1.481 +2.067j ]
  R*X*    = [ 0.8781+0.9344j  1.3772+1.4543j  5.5021-0.2497j  0.9908-1.7171j
 -0.086 -2.4836j  1.481 +2.067j ]
  Match? True

DFT 4: R*x <-> R*X
  fft(Rx) = [ 0.8781-0.9344j  1.3772-1.4543j  5.5021+0.2497j  0.9908+1.7171j
 -0.086 +2.4836j  1.481 -2.067j ]
  R*X     = [ 0.8781-0.9344j  1.3772-1.4543j  5.5021+0.2497j  0.9908+1.7171j
 -0.086 +2.4836j  1.481 -2.067j ]
  Match? True

Example 13:
x  = [ 1.+0.j  0.+1.j -2.+0.j  0.+2.j]
X  = [-1.+3.j  2.+0.j -1.-3.j  4.+0.j]
Expected: [-1+3j, 2, -1-3j, 4]

x* = [ 1.-0.j  0.-1.j -2.-0.j  0.-2.j]
R*X* = [-1.-3.j  4.+0.j -1.+3.j  2.+0.j]
fft(x*) = [-1.-3.j  4.+0.j -1.+3.j  2.+0.j]

Rx = [ 1.+0.j  0.+2.j -2.+0.j  0.+1.j]
R*X = [-1.+3.j  4.+0.j -1.-3.j  2.+0.j]
fft(Rx) = [-1.+3.j  4.+0.j -1.-3.j  2.+0.j]


---
## Summary

| Matrix | Effect | Property |
|---|---|---|
| $P$ | Circular shift down 1 | $P^N=I$, permutation |
| $R$ | Circular reversal | $R^2=I$, $R^T=R$ |
| $F^k$ | Entry-wise multiply by $v^{(k)}$ | Diagonal |

| DFT Property | Time → Frequency |
|---|---|
| DFT 3 | $x^* \leftrightarrow RX^*$ |
| DFT 4 | $Rx \leftrightarrow RX$ |

### Mistakes to Avoid
1. $Px$ shifts **down** (last entry moves to top). A shift **up** is $P^{-1}x=P^{N-1}x$.
2. DFT 3 applies **both** conjugation and reversal to $X$. Don't forget one or the other.
3. The modulation matrix $F^k$ is **diagonal**, NOT a permutation matrix.
